# Training a Machine Model to predict Placement Status of a Student according to Academic records

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("data/student_career_success_dataset.csv")
df.head()

,Student_ID,Age,Gender,University_Year,Major,Attendance_Percentage,Study_Hours_Per_Week,CGPA,Academic_Performance,Programming_Skill,...,Teamwork,Problem_Solving,English_Proficiency,Interview_Score,Employability_Score,Placement_Status,Company_Tier,Career_Field,Placement_Mode,Starting_Salary_USD
0,ST000001,22,Male,Sophomore,Computer Science,86,21,3.28,Good,9,...,7,8,Advanced,72,231.70,Placed,Tier 1,Backend Developer,Campus Placement,72082
1,ST000002,20,Female,Freshman,Computer Science,64,18,2.53,Poor,6,...,6,5,Basic,59,157.45,Not Placed,No Company,Not Placed,Not Applicable,0
2,ST000003,23,Male,Senior,Information Technology,75,19,2.83,Average,8,...,8,7,Advanced,76,203.45,Placed,Tier 1,IT Support Engineer,Campus Placement,94898
3,ST000004,23,Female,Senior,Information Technology,80,19,2.80,Average,5,...,6,6,Intermediate,67,171.50,Not Placed,No Company,Not Placed,Not Applicable,0
4,ST000005,23,Female,Senior,Software Engineering,73,19,2.93,Average,9,...,8,9,Advanced,82,225.45,Placed,Tier 1,Software Engineer,Internship Conversion,85256


In [3]:
df.columns

Index(['Student_ID', 'Age', 'Gender', 'University_Year', 'Major',
       'Attendance_Percentage', 'Study_Hours_Per_Week', 'CGPA',
       'Academic_Performance', 'Programming_Skill', 'Projects_Completed',
       'Certifications', 'Hackathons', 'GitHub_Profile', 'Internships',
       'Leadership_Experience', 'LinkedIn_Profile', 'Resume_Score',
       'Communication_Skills', 'Teamwork', 'Problem_Solving',
       'English_Proficiency', 'Interview_Score', 'Employability_Score',
       'Placement_Status', 'Company_Tier', 'Career_Field', 'Placement_Mode',
       'Starting_Salary_USD'],
      dtype='str')

In [4]:
X = df.drop(columns=['Student_ID','Placement_Status', 'Company_Tier', 'Career_Field', 'Placement_Mode','Starting_Salary_USD'])
y = df["Placement_Status"].map({"Placed":1,"Not Placed":0})

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [6]:
from sklearn.metrics import precision_score, accuracy_score, recall_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model):
    y_pred = model.predict(X_test)
    print("Model Evaluation: ")
    print(f"Precision Score : {precision_score(y_test,y_pred)*100:.3f} %")
    print(f"Accuracy Score : {accuracy_score(y_test,y_pred)*100:.3f} %")
    print(f"Recall Score : {recall_score(y_test,y_pred)*100:.3f} %")
    print(f"F1 Score : {f1_score(y_test,y_pred)*100:.3f} %")
    print(f"Classification Report : \n{classification_report(y_test,y_pred)}")
    print(f"Confusion Matrix : \n{confusion_matrix(y_test,y_pred)}")

In [7]:
cat_cols = X.select_dtypes(include="str").columns
num_cols = X.select_dtypes(exclude="str").columns

In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
scaler = StandardScaler()
ohe = OneHotEncoder()

In [9]:
from sklearn.compose import ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num",scaler,num_cols),
        ("cat",ohe,cat_cols)
    ]
)

In [10]:
from sklearn.pipeline import Pipeline
def into_pipe(model):
    pipe=Pipeline(
        steps=[
            ("preprocess",preprocess),
            ("model",model)
        ]
    )
    return pipe

In [17]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr_pipe=into_pipe(lr)
lr_pipe.fit(X_train,y_train)
evaluate_model(lr_pipe)

Model Evaluation: 
Precision Score : 84.179 %
Accuracy Score : 81.890 %
Recall Score : 94.582 %
F1 Score : 89.078 %
Classification Report : 
              precision    recall  f1-score   support

           0       0.66      0.37      0.47      2192
           1       0.84      0.95      0.89      7808

    accuracy                           0.82     10000
   macro avg       0.75      0.66      0.68     10000
weighted avg       0.80      0.82      0.80     10000

Confusion Matrix : 
[[ 804 1388]
 [ 423 7385]]


### Logistic Regression — Results

Logistic Regression achieved an accuracy of 81.89% and an F1-score of 89.08% for the Placed class.

The model performs substantially better on the Placed class than on the Not Placed class. For Placed students, the model achieves 94.58% recall, correctly identifying most students who were actually placed.

However, the recall for Not Placed students is only 37%. Out of 2,192 students who were actually not placed, the model correctly identified 804 and incorrectly classified 1,388 as placed.

This indicates that the model has difficulty identifying the minority Not Placed class and tends to predict Placed more frequently.

In [16]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb_pipe = into_pipe(gnb)
gnb_pipe.fit(X_train,y_train)
evaluate_model(gnb_pipe)

Model Evaluation: 
Precision Score : 88.005 %
Accuracy Score : 76.670 %
Recall Score : 81.186 %
F1 Score : 84.458 %
Classification Report : 
              precision    recall  f1-score   support

           0       0.47      0.61      0.53      2192
           1       0.88      0.81      0.84      7808

    accuracy                           0.77     10000
   macro avg       0.68      0.71      0.69     10000
weighted avg       0.79      0.77      0.78     10000

Confusion Matrix : 
[[1328  864]
 [1469 6339]]


### Naive Bayes — Results

Naive Bayes achieved an accuracy of 76.67% and an F1-score of 84.46% for the Placed class.

Compared with Logistic Regression, Naive Bayes achieved substantially higher recall for the Not Placed class (61% vs. 37%), correctly identifying more students who were actually not placed. However, its precision for this class was lower (47%), resulting in a larger number of students being incorrectly classified as Not Placed.

Overall, Naive Bayes produced lower accuracy and Placed-class performance than Logistic Regression, but it showed a different behavior in handling the minority Not Placed class.

In [38]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=9)
knn_pipe = into_pipe(knn)
knn_pipe.fit(X_train,y_train)
evaluate_model(knn_pipe)

Model Evaluation: 
Precision Score : 83.896 %
Accuracy Score : 80.600 %
Recall Score : 93.007 %
F1 Score : 88.217 %
Classification Report : 
              precision    recall  f1-score   support

           0       0.59      0.36      0.45      2192
           1       0.84      0.93      0.88      7808

    accuracy                           0.81     10000
   macro avg       0.72      0.65      0.67     10000
weighted avg       0.79      0.81      0.79     10000

Confusion Matrix : 
[[ 798 1394]
 [ 546 7262]]


In [37]:
from sklearn.model_selection import GridSearchCV
params = {
    "model__n_neighbors":[3,5,7,9,11]
}
grid = GridSearchCV(
    estimator=knn_pipe,
    param_grid=params,
    cv=5,
    scoring="f1_macro"
)
grid.fit(X_train,y_train)
print("Best parameters : ", grid.best_params_)

Best parameters :  {'model__n_neighbors': 9}


### KNN — Results

K-Nearest Neighbors (KNN) was evaluated as a baseline model after applying the preprocessing pipeline. Since KNN relies on distances between observations, numerical features were standardized before calculating distances.

The baseline KNN model achieved an accuracy of 80.60% and an F1-score of 88.22% for the Placed class. The model performed substantially better on the Placed class than on the Not Placed class. The recall for Not Placed was only 36%, meaning that most students who were actually not placed were incorrectly classified as placed.

The model's behavior is similar to Logistic Regression, with strong performance on the majority Placed class but limited ability to identify the minority Not Placed class.

#### Hyperparameter Tuning

To investigate whether KNN's performance could be improved, GridSearchCV was used to search over different values of n_neighbors, distance weighting, and distance metrics. Macro F1-score was used as the optimization metric so that both classes contributed equally despite the class imbalance. The best configuration found was: n_neighbors = 9

In [52]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(
    n_estimators=201,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=5,
    oob_score=True,
    random_state=42
)
rfc_pipe = into_pipe(rfc)
rfc_pipe.fit(X_train,y_train)
evaluate_model(rfc_pipe)
print("OOB Score : ",rfc.oob_score_)

Model Evaluation: 
Precision Score : 84.223 %
Accuracy Score : 81.830 %
Recall Score : 94.416 %
F1 Score : 89.028 %
Classification Report : 
              precision    recall  f1-score   support

           0       0.65      0.37      0.47      2192
           1       0.84      0.94      0.89      7808

    accuracy                           0.82     10000
   macro avg       0.75      0.66      0.68     10000
weighted avg       0.80      0.82      0.80     10000

Confusion Matrix : 
[[ 811 1381]
 [ 436 7372]]
OOB Score :  0.815975


### Random Forest — Results

Random Forest achieved an accuracy of 81.82%, with an F1-score of 89.01% for the Placed class. Similar to Logistic Regression and KNN, it performed substantially better on the Placed class than on the Not Placed class. The recall for Not Placed was 37%, with 821 out of 2,192 actually not-placed students correctly identified. The model achieved an OOB score of 81.59%, which was close to its test accuracy of 81.82%. Overall, Random Forest showed performance similar to the Logistic Regression baseline, without substantially improving identification of the minority class.